In [29]:
import spacy

nlp = spacy.load("pl_core_news_lg")

In [30]:
import pandas as pd


gold_answers_train = pd.read_json("./data/convos.jsonl", lines=True)[
    "reformulated question"
].to_list()
gold_answers_val = pd.read_json("./data/rag_paraphrase_eval.jsonl", lines=True)[
    "gold_paraphrase"
].to_list()

gold_answers = [*gold_answers_train, *gold_answers_val]

In [31]:
gold_nlp_docs = [doc for doc in nlp.pipe(gold_answers)]

In [32]:
from collections.abc import Callable, Sequence
import math

import numpy as np
from spacy.ml import Doc
from spacy.tokens import Token


def is_valid_word_token(token: Token):
    if token.is_punct:
        return False

    if token.is_stop:
        return False

    if token.like_num:
        return False

    return token.pos_ in ["NOUN", "PROPN", "ADJ", "VERB", "ADV"]


def is_oov_token(token: Token):
    return token.is_oov


def select_lemmas_by(
    docs: Sequence[Doc], token_selector: Callable[[Token], bool] = is_valid_word_token
):
    lemmas_sets = get_lemmas_sets(docs, token_selector)
    lemmas_set = set().union(*lemmas_sets)

    return lemmas_set


def get_lemmas_sets(docs: Sequence[Doc], token_selector: Callable[[Token], bool]):
    lemmas_sets = [
        {token.lemma_ for token in doc if token_selector(token)} for doc in docs
    ]
    return lemmas_sets


lemmas_per_doc = get_lemmas_sets(gold_nlp_docs, token_selector=is_valid_word_token)

lemmas_df = pd.DataFrame(
    select_lemmas_by(gold_nlp_docs, is_valid_word_token), columns=["lemma"]
)
lemmas_df["df"] = lemmas_df["lemma"].apply(
    lambda l: sum(l in lemmas_set for lemmas_set in lemmas_per_doc)
)
N = len(lemmas_df)
lemmas_df["idf"] = lemmas_df["df"].apply(lambda df: math.log(N / (df + 1)))

oov_weight = np.percentile(lemmas_df["idf"], 95).item()

In [33]:
lemmas_df.set_index("lemma")["idf"]

lemma
panieński       5.402677
projekt         5.402677
składać         4.709530
paszportowy     5.402677
młodociany      5.402677
                  ...   
podręcznik      4.997212
zastrzeżenie    5.402677
ewidencyjny     4.709530
wizyta          4.149914
być             5.402677
Name: idf, Length: 444, dtype: float64

In [ ]:
from typing import Any

from pandas import Series
from spacy.language import Language


class IDFWeightedSoftTermF1:
    r"""
Computes an IDF-weighted term-level F1 score between a predicted query and a
gold query.

This metric is intended for evaluating query paraphrases used for retrieval.
It compares the predicted paraphrase ``predicted`` against the reference
paraphrase ``expected`` after linguistic normalization with spaCy. Both texts
are converted into sets of content lemmas, and the score is computed as an
IDF-weighted F1 over exact lemma matches.

Let:

.. math::

    A &= \operatorname{terms}(\text{predicted}) \\
    G &= \operatorname{terms}(\text{expected}) \\
    C &= A \cap G

where :math:`A` is the set of lemmas extracted from the model output,
:math:`G` is the set of lemmas extracted from the gold query, and :math:`C`
is the set of exact lemma matches.

The function ``terms(.)`` keeps only selected content tokens and removes
punctuation, stop words, numeric-like tokens, and tokens whose POS tag is not
one of:

.. math::

    \{\text{NOUN}, \text{PROPN}, \text{ADJ}, \text{VERB}, \text{ADV}\}

Lemma weights are estimated from the full list of gold queries passed to the
constructor. Each gold query :math:`g_i` is treated as one document. For a
lemma :math:`t`, document frequency and smoothed IDF are defined as:

.. math::

    \operatorname{df}(t)
        &= |\{g_i : t \in \operatorname{terms}(g_i)\}| \\
    w(t)
        &= \log\left(\frac{N + 1}{\operatorname{df}(t) + 1}\right) + 1

where :math:`N` is the number of gold queries.

Lemmas absent from the IDF lookup table are treated as out-of-vocabulary terms
and assigned a fixed OOV weight:

.. math::

    w_{\text{OOV}} = P_{95}(\{w(t) : t \in V\})

where :math:`V` is the vocabulary of content lemmas observed in
``gold_questions_list``. This prevents unseen predicted terms from being
ignored in precision. In particular, unsupported or hallucinated terms
produced by the model still increase the precision denominator.

Weighted precision, recall, and F1 are computed as:

.. math::

    P &=
    \frac{\sum_{t \in C} w(t)}
         {\sum_{t \in A} w(t)} \\[4pt]
    R &=
    \frac{\sum_{t \in C} w(t)}
         {\sum_{t \in G} w(t)} \\[4pt]
    F_1 &=
    \frac{2PR}{P + R}

where :math:`w(t)` is taken from the IDF lookup table when available and is
otherwise set to :math:`w_{\text{OOV}}`.

The returned feedback string reports the asymmetric differences between the
gold query and the prediction:

.. math::

    \operatorname{missing} &= G \setminus A \\
    \operatorname{spurious} &= A \setminus G

Missing terms are terms expected in the gold query but absent from the model
output. Spurious terms are terms produced by the model but absent from the gold
query. These diagnostics make the metric usable in reflective prompt
optimization loops such as GEPA.

Args:
    gold_questions_list:
        List of gold paraphrases used to construct the IDF lookup table. Each
        item is processed with ``nlp`` and contributes one document for
        document-frequency estimation.
    nlp:
        spaCy language pipeline used for tokenization, POS tagging, stop-word
        detection, and lemmatization. For Polish, this is expected to be a
        Polish model such as ``pl_core_news_lg``.

Returns:
    Callable metric object. Calling the object with ``predicted`` and
    ``expected`` returns:

    ``tuple[float, str]``:
        - ``score``: IDF-weighted exact-lemma F1 score in the range
          ``[0.0, 1.0]`` under non-negative IDF weights.
        - ``feedback``: textual diagnostic feedback containing missing and
          spurious lemmas.

Notes:
    Despite the class name, this implementation currently uses hard lemma
    matching only. It does not yet perform embedding-based soft matching.

    Predicted OOV terms are penalized through the precision denominator. This
    is intentional: unseen terms in the model output may represent unsupported
    additions or hallucinated constraints.
"""

    def __init__(self, gold_questions_list: list[str], nlp: Language):
        self._nlp = nlp
        self._lemma_weight_lut = self._construct_lemma_weight_lut(gold_questions_list)
        if len(self._lemma_weight_lut) == 0:
            self._oov_weight = 1.0
        else:
            self._oov_weight = float(
                np.percentile(self._lemma_weight_lut.to_numpy(), 95)
            )

    def _construct_lemma_weight_lut(self, gold_questions_list: list[str]) -> Series:
        """Returns Pandas series with index `lemma` (lemma) and value `idf` (IDF for lemma, calculated for entire `gold_questions_list`)"""
        docs = [doc for doc in self._nlp.pipe(gold_questions_list)]

        lemmas_per_doc: list[set[str]] = [self._lemmatize_terms(doc) for doc in docs]
        lemmas_total: set[str] = set().union(*lemmas_per_doc)

        n_docs = len(gold_questions_list)

        lemmas_df = pd.DataFrame(lemmas_total, columns=["lemma"])
        lemmas_df["df"] = lemmas_df["lemma"].apply(
            lambda l: sum(l in lemmas_set for lemmas_set in lemmas_per_doc)
        )
        lemmas_df["idf"] = lemmas_df["df"].apply(
            lambda df: math.log((n_docs + 1) / (df + 1)) + 1
        )

        return lemmas_df.set_index("lemma")["idf"]

    def __call__(self, predicted: str, expected: str) -> Any:
        predicted_lemmatized = self._lemmatize_terms(predicted)
        expected_lemmatized = self._lemmatize_terms(expected)

        common_lemmas = predicted_lemmatized.intersection(expected_lemmatized)

        common_lemmas_wt_sum = self._sum_weights(common_lemmas)
        predicted_lemmas_wt_sum = self._sum_weights(predicted_lemmatized)
        expected_lemmas_wt_sum = self._sum_weights(expected_lemmatized)

        if predicted_lemmas_wt_sum == 0 and expected_lemmas_wt_sum == 0:
            return 1.0, "Brak rozbieżności w ważnych terminach"

        if predicted_lemmas_wt_sum == 0 or expected_lemmas_wt_sum == 0:
            return 0.0, "Jedna z parafraz nie zawiera żadnych terminów treściowych."

        precision = common_lemmas_wt_sum / predicted_lemmas_wt_sum
        recall = common_lemmas_wt_sum / expected_lemmas_wt_sum

        if precision + recall == 0:
            score = 0.0
        else:
            score = 2 * precision * recall / (precision + recall)

        missing_terms = expected_lemmatized.difference(predicted_lemmatized)
        spurious_terms = predicted_lemmatized.difference(expected_lemmatized)

        feedbacks = []
        if missing_terms:
            feedbacks.append(
                f"Brakujące ważne terminy: {self._format_terms_by_weight(missing_terms)}."
            )
        if spurious_terms:
            feedbacks.append(
                f"Nadmiarowe terminy: {self._format_terms_by_weight(spurious_terms)}."
            )

        if feedbacks:
            feedback_str = "\n".join(feedbacks)
        else:
            feedback_str = "Brak rozbieżności w ważnych terminach"

        return float(score), feedback_str

    def _lemmatize_terms(self, doc: Doc | str):
        if isinstance(doc, str):
            doc = self._nlp(doc)
        return {token.lemma_ for token in doc if self.is_term(token)}

    @staticmethod
    def is_term(token: Token):
        if token.is_punct:
            return False

        if token.is_stop:
            return False

        if token.like_num:
            return False

        return token.pos_ in ["NOUN", "PROPN", "ADJ", "VERB", "ADV"]

    def _sum_weights(self, lemmas: set[str]) -> float:
        return float(sum(self._get_weight(lemma) for lemma in lemmas))

    def _get_weight(self, lemma: str) -> float:
        """Gets the lemma weight with fallback to OOV weight"""
        if lemma in self._lemma_weight_lut.index:
            return float(self._lemma_weight_lut.loc[lemma])
        return self._oov_weight

    def _format_terms_by_weight(self, terms: set[str], limit: int = 10) -> str:
        sorted_terms = sorted(terms, key=self._get_weight, reverse=True)
        limited_terms = sorted_terms[:limit]

        grouped_terms: dict[float, list[str]] = {}

        for term in limited_terms:
            weight = round(self._get_weight(term), 2)
            grouped_terms.setdefault(weight, []).append(term)

        formatted_groups = []
        for weight, group_terms in grouped_terms.items():
            terms_str = ", ".join(sorted(group_terms))
            formatted_groups.append(f"{terms_str} ({weight:.2f})")

        suffix = ""
        if len(sorted_terms) > limit:
            suffix = f" oraz {len(sorted_terms) - limit} więcej"

        return ", ".join(formatted_groups) + suffix


metric = IDFWeightedSoftTermF1(gold_questions_list=gold_answers, nlp=nlp)

metric(
    "Jak załatwić we Wrocławiu sprawę: dodatek mieszkaniowy dla gospodarstwa domowego? Chodzi mi o dokumenty, opłaty i miejsce złożenia wniosku.",
    "Jakie dokumenty i opłaty do dodatku mieszkaniowego i gdzie złożyć wniosek katastralny?",
)


(0.5193632337814725,
 'Brakujące ważne terminy: katastralny (5.52).\nNadmiarowe terminy: chodzić, domowy, gospodarstwo, miejsce (5.52), sprawa (4.82), załatwić (3.72), Wrocław (1.73).')

In [37]:
%%timeit
metric(
    "Jak załatwić we Wrocławiu sprawę: dodatek mieszkaniowy dla gospodarstwa domowego? Chodzi mi o dokumenty, opłaty i miejsce złożenia wniosku.",
    "Jakie dokumenty i opłaty do dodatku mieszkaniowego i gdzie złożyć wniosek katastralny?",
)


12.8 ms ± 418 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
